# 01_04 Our model as a black box

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **the basics** (chapter 1): the ideas every stage uses
>
> Project files this connects to:
>
> - `src/homer_ai/model.py`
> - `src/homer_ai/homer_model.pt`
> - `src/homer_ai/homer_mcp_server.py`
>
> **This notebook:** Chapters 2-5 build and train `HomerGreekLLM`. Seeing first what it does from outside, untrained and trained, shows what all the parts are for.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- What our model does, seen from outside: text in, scores for the next character out
- What an untrained model writes, and what a trained one writes
- Which parts of the box the next chapters open

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **black box** | something you use by its inputs and outputs, without looking inside | a TV remote |
| **score** | the number the model gives each possible next character; higher means more likely | 137 scores per position |
| **checkpoint** | a saved copy of a trained model's weights, in a file | `homer_model.pt` |

## 3. The idea

### ❓ Why use the model without understanding it first?

Because then you know what all the parts are for. It's like driving a car before you open up the engine: first you see what it does, and the later chapters show how. We open the box in chapters 2-4.

### ❓ What goes in, and what comes out?

**In:** a line of text, turned into numbers by the tokenizer (chapter 2).

**Out:** for every position in the line, 137 scores, one for each character the model knows (its **vocabulary**). The highest score is the model's guess for the next character.

### ❓ Where does a trained model come from?

Training (chapter 5) adjusts the weights and saves them in a **checkpoint** file, `src/homer_ai/homer_model.pt`. Your own file isn't stored in git, so a fresh copy of the project doesn't have one. What it does have is a ready-made one, [models/odyssey_book1.pt](../../models/README.md), and `load_homer_model` uses that and tells you so. To make your own, run `uv run homer-train` (about 8 to 10 minutes). After that, yours gets used instead. (The course has a second, bigger model too, the GPT of chapter 4. Its command is `uv run homer-train --model gpt`, about 3 minutes, and [05_07](../05_training_on_the_odyssey/05_07_train_the_gpt.ipynb) walks through it.)

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ How do I load an untrained and a trained model?

The untrained one is just a new `HomerGreekLLM`, with random weights. The trained one loads its weights from the checkpoint and builds its tokenizer from what's saved in the same file, exactly as `homer_mcp_server.py` does. That matters: the model learned with one numbering of the characters (`π` = 41, say), and a tokenizer built from another text would number them differently, so the trained weights would read nonsense. The helper `load_homer_model` (in `src/homer_ai/lessons.py`) does all this without writing anything. [05_06](../05_training_on_the_odyssey/05_06_saving_and_loading.ipynb) builds the steps by hand.

👉 **Predict first:** the cell prints one line saying which model it loaded. On your computer, will it be your own or the ready-made one?

In [ ]:
import torch

from homer_ai.lessons import load_homer_model
from homer_ai.model import HomerGreekLLM

# read-only: your homer_model.pt, or else the ready-made one
tokenizer, trained = load_homer_model()

torch.manual_seed(42)
untrained = HomerGreekLLM(tokenizer.vocab_size, embed_dim=32)
untrained.eval()  # eval(): "just guessing" mode, not training

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

One line telling you which model was loaded: `Using the ready-made model from models/ (137 tokens)…` on a fresh copy of the project, or `Using your own model…` once you've trained one. Either way the tokenizer has 137 tokens, taken from the file.

</details>

### ❓ What comes out of the box?

Give the model `ἄνδρα μο` (8 characters) and it gives back a table: 8 rows (one per position) × 137 scores. `softmax` then turns the last row of scores into **probabilities** that add up to 1.

```
ἄ  ν  δ  ρ  α  ' '  μ  ο     →   model   →   8 rows × 137 scores
                                             last row → chances for the character after `ἄνδρα μο`
```

👉 **Predict first:** for the trained model, which character after `ἄνδρα μο`?

In [ ]:
ids = torch.tensor(tokenizer.encode("ἄνδρα μο"), dtype=torch.long)

for name, model in [("untrained", untrained), ("trained", trained)]:
    if model is None:
        continue
    with torch.no_grad():
        scores = model(ids)
    chances = torch.softmax(scores[-1], dim=-1)
    top = torch.topk(chances, 5)
    best = [
        (tokenizer.decode([i.item()]), round(c.item(), 3))
        for c, i in zip(top.values, top.indices, strict=True)
    ]
    print(name, tuple(scores.shape), best)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The table is `(8, 137)`. The untrained model gives every character about the same tiny chance (about 0.009 each, close to 1 in 137, so a pure guess). The trained one puts `ι` first, at about 0.21. It has learned that `ἄνδρα μοι` comes next.

</details>

### ❓ Can I see the trained model's chances?

Yes. Here's a bar chart of its 8 best guesses for the character after `ἄνδρ` (the start of `ἄνδρα`, "man"). Each bar is a probability, and all 137 of them add up to 1.

👉 **Predict first:** will one bar tower over the rest, or will several be close?

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import ACCENT, INK, MUTED, studio_axes, studio_figure

if trained is None:
    print("no trained model yet: make one first, then run this cell again")
else:
    ids = torch.tensor(tokenizer.encode("ἄνδρ"), dtype=torch.long)
    with torch.no_grad():
        chances = torch.softmax(trained(ids)[-1], dim=-1)
    top = torch.topk(chances, 8)
    letters = [repr(tokenizer.decode([i.item()])) for i in top.indices]
    values = top.values.tolist()

    fig = studio_figure(
        "The trained model's chances",
        f"After ἄνδρ, its best guess is {letters[0]}: {values[0]:.0%}",
    )
    ax = studio_axes(fig)
    ax.set_axisbelow(True)  # grid lines behind the bars
    bars = ax.bar(letters, values, color=MUTED)
    bars[0].set_color(ACCENT)
    for bar, value in zip(bars, values, strict=True):
        ax.text(
            bar.get_x() + bar.get_width() / 2, value + 0.01, f"{value:.2f}",
            ha="center", color=INK, family="monospace", fontsize=11,
        )
    ax.tick_params(axis="x", labelsize=14, labelcolor=INK)
    ax.grid(axis="x", visible=False)
    ax.set_ylabel("probability")
    plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

No bar towers over the rest. The best guess is only about 0.22, and `ε` is close behind. So the model hands back a table of chances, not one fixed answer. It really is guessing, and Greek really can go on in several ways.

</details>

## 5. Full example

### ❓ What does each model write?

We start with `ἄνδρα` and do this 40 times: guess the most likely next character, add it, guess again.

👉 **Predict first:** will the trained model write real Homer?

In [ ]:
def write(model, tokenizer, prompt, n_new):
    """Guess the next character, add it, repeat: always the most likely one."""
    ids = tokenizer.encode(prompt)
    with torch.no_grad():  # just guessing, not learning
        for _ in range(n_new):
            scores = model(torch.tensor(ids[-24:], dtype=torch.long))
            best = torch.argmax(scores[-1])  # argmax: where the highest score is
            ids.append(best.item())
    return tokenizer.decode(ids)


print("untrained:", write(untrained, tokenizer, "ἄνδρα", 40))
if trained is not None:
    print("trained:  ", write(trained, tokenizer, "ἄνδρα", 40))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

With the model trained in this project (yours may differ if you retrained), you get:

```
untrained: ἄνδραἂ<|unk|><|unk|><|unk|>ὮὮὮὮὮὮὮὮὮὮὮὮὮὮὮὮὮὮῶὧῶὄχῶὄὧὧῶὄῶἣὄῶἣἣἣ
trained:   ἄνδρα τοι και και και κα κα κατον κατοι και κ
```

The untrained model writes random characters, even `<|unk|>` (the stand-in for a character it doesn't know). The trained one writes Greek-looking words with spaces, but then gets stuck repeating `και`, just like the counting model in 01_01. Fixing that is what chapters 2-6 are for.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Is this the same model that Claude talks to?

It can be, but it isn't the first choice. `homer_mcp_server.py` looks for the GPT of chapter 4 first (yours, then the ready-made one), and only uses this small model if there's no GPT file, or if you point `HOMER_MODEL` at it ([06_03](../06_making_it_talk/06_03_the_mcp_server.ipynb)). Whichever it loads, it runs this same guessing loop whenever Claude calls the tool.

## 6. Recap

- From outside, the model is text in → 137 scores per position out; the highest score is its guess.
- Untrained, it guesses at random; trained on Book 1, it writes Greek-looking words but repeats itself.
- Chapters 2-4 open the box: tokens and embeddings, attention, the full model.

**You can now:**

- load the model and give it a text
- read what comes out: one row of 137 scores per position
- tell an untrained model's writing from a trained one's

**Next:** Chapter 2: [Text to numbers](../02_text_to_numbers/02_00_chapter_overview.ipynb).